##Install dependencies  

In [35]:
!pip install dash --quiet

##Library imports

In [36]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from dash import dcc, html, Input, Output
import ipywidgets as widgets
from IPython.display import display, clear_output

##Mount Google Drive

In [37]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = '/content/drive/MyDrive/CIS9655_Group12/data/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [38]:
import os

# Check if the base path exists
base_path = '/content/drive/MyDrive/CIS9655_Group12/data/'
print(os.listdir(base_path))

['selected_us_stocks_daily_by_sector_2025', 'gdelt_gkg_tone_timeline.csv', 'wikimedia_pageviews.csv']


In [39]:
tech_folder = '/content/drive/MyDrive/CIS9655_Group12/data/selected_us_stocks_daily_by_sector_2025/technology/'
print(os.listdir(tech_folder))

['QQQ.csv', 'MSFT.csv', 'ORCL.csv', 'XLK.csv', 'GOOGL.csv', 'AMZN.csv', 'META.csv', 'AAPL.csv', 'CRM.csv']


##Load stock CSVs

In [40]:
WINDOW_START = '2025-05-01'
WINDOW_END   = '2025-06-15'

TICKERS = ['MSFT', 'AAPL', 'AMZN', 'GOOGL', 'META',
            'ORCL', 'CRM', 'QQQ', 'XLK']

stocks = {}
for ticker in TICKERS:
    df = pd.read_csv(f'{DATA_PATH}selected_us_stocks_daily_by_sector_2025/technology/{ticker}.csv', parse_dates=['date'])
    mask = (df['date'] >= WINDOW_START) & (df['date'] <= WINDOW_END)
    stocks[ticker] = df.loc[mask].reset_index(drop=True)

msft = stocks['MSFT']

##Load GDELT tone data

In [41]:
gdelt = pd.read_csv(f'{DATA_PATH}gdelt_gkg_tone_timeline.csv')
gdelt = gdelt[['Date', 'Value']].sort_values('Date').reset_index(drop=True)
gdelt = gdelt.rename(columns={'Date': 'date', 'Value': 'tone'})

##Load Wikimedia pageviews

In [42]:
# Wikipedia data
wiki = pd.read_csv(f'{DATA_PATH}wikimedia_pageviews.csv')
wiki['date'] = wiki['timestamp'].astype(str).str[:8]  # Extract YYYYMMDD from timestamp
wiki = wiki[['date', 'article', 'views']].sort_values('date').reset_index(drop=True)

##Sanity check

In [43]:
print("=== Stocks loaded ===")
for t, df in stocks.items():
    print(f"  {t}: {len(df)} rows  |  {df['date'].min().date()} → {df['date'].max().date()}")

print(f"\n=== GDELT: {len(gdelt)} rows ===")
print(gdelt.head(3).to_string(index=False))

print(f"\n=== Wikimedia: {len(wiki)} rows ===")
print(wiki.head(3).to_string(index=False))

=== Stocks loaded ===
  MSFT: 31 rows  |  2025-05-01 → 2025-06-13
  AAPL: 31 rows  |  2025-05-01 → 2025-06-13
  AMZN: 31 rows  |  2025-05-01 → 2025-06-13
  GOOGL: 31 rows  |  2025-05-01 → 2025-06-13
  META: 31 rows  |  2025-05-01 → 2025-06-13
  ORCL: 31 rows  |  2025-05-01 → 2025-06-13
  CRM: 31 rows  |  2025-05-01 → 2025-06-13
  QQQ: 31 rows  |  2025-05-01 → 2025-06-13
  XLK: 31 rows  |  2025-05-01 → 2025-06-13

=== GDELT: 45 rows ===
      date   tone
2025-05-01 0.4496
2025-05-02 0.0058
2025-05-03 0.5408

=== Wikimedia: 46 rows ===
    date   article  views
20250501 Microsoft   8156
20250502 Microsoft   7543
20250503 Microsoft   7097


##Head of few CSV's

In [44]:
for t, df in stocks.items():
    print(f"\n=== {t} ===")
    print(df.head())

print("\n=== GDELT ===")
print(gdelt.head())

print("\n=== Wikimedia ===")
print(wiki.head())


=== MSFT ===
        date ticker    volume    open   close    high       low  \
0 2025-05-01   MSFT  58938100  431.11  425.40  436.99  424.9000   
1 2025-05-02   MSFT  30757434  431.74  435.28  439.44  429.9850   
2 2025-05-05   MSFT  20136053  432.87  436.17  439.50  432.1100   
3 2025-05-06   MSFT  15104204  432.20  433.31  437.73  431.1700   
4 2025-05-07   MSFT  23307241  433.84  433.35  438.12  431.1103   

          window_start  transactions  
0  1746072000000000000        913598  
1  1746158400000000000        542335  
2  1746417600000000000        328657  
3  1746504000000000000        297624  
4  1746590400000000000        330269  

=== AAPL ===
        date ticker     volume    open   close    high     low  \
0 2025-05-01   AAPL   57365675  209.08  213.32  214.56  208.90   
1 2025-05-02   AAPL  101010621  206.09  205.35  206.99  202.16   
2 2025-05-05   AAPL   69018452  203.10  198.89  204.10  198.21   
3 2025-05-06   AAPL   51216482  198.21  198.51  200.65  197.02   
4 202

# **Full Dashboard**

In [45]:
# =========================
# BLOCK 1: PREPARE DATA
# NO HARD-CODED PEAK DATES
# =========================

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# -----------------------------
# 1) Clean stock dates
# -----------------------------
for t in stocks:
    stocks[t] = stocks[t].copy()
    stocks[t]['date'] = pd.to_datetime(stocks[t]['date'])

msft = stocks['MSFT'].copy()

# -----------------------------
# 2) Clean GDELT
# -----------------------------
gdelt_fixed = gdelt.copy()
gdelt_fixed['date'] = pd.to_datetime(gdelt_fixed['date'], errors='coerce')

gdelt_daily = (
    gdelt_fixed.groupby('date', as_index=False)['tone']
    .mean()
    .sort_values('date')
)

# -----------------------------
# 3) Reload Wikipedia correctly
# -----------------------------
wiki_raw = pd.read_csv(f'{DATA_PATH}wikimedia_pageviews.csv').copy()
wiki_raw['timestamp'] = wiki_raw['timestamp'].astype(str)
wiki_raw['date_str'] = wiki_raw['timestamp'].str[:8]
wiki_raw['date'] = pd.to_datetime(wiki_raw['date_str'], format='%Y%m%d', errors='coerce')

wiki_fixed = wiki_raw[['date', 'article', 'views']].copy()
wiki_fixed = wiki_fixed.dropna(subset=['date'])
wiki_fixed = wiki_fixed[
    (wiki_fixed['date'] >= pd.to_datetime(WINDOW_START)) &
    (wiki_fixed['date'] <= pd.to_datetime(WINDOW_END))
]

wiki_daily = (
    wiki_fixed.groupby('date', as_index=False)['views']
    .sum()
    .sort_values('date')
)

# -----------------------------
# 4) Build main merged dataframe
# -----------------------------
base = msft[['date', 'close']].copy()
base = base.rename(columns={'close': 'msft_close'})
base['msft_return'] = base['msft_close'].pct_change() * 100

base = base.merge(gdelt_daily, on='date', how='left')
base = base.merge(wiki_daily, on='date', how='left')

base['tone'] = base['tone'].interpolate(limit_direction='both')
base['views'] = base['views'].fillna(0)
base['views_ma3'] = base['views'].rolling(3, min_periods=1).mean()

# -----------------------------
# 5) Build normalized comparison dataframe
# -----------------------------
comparison_tickers = ['MSFT', 'QQQ']
norm_df = pd.DataFrame({'date': stocks['MSFT']['date']})

for t in comparison_tickers:
    temp = stocks[t][['date', 'close']].copy().sort_values('date')
    temp[f'{t}_norm'] = temp['close'] / temp['close'].iloc[0] * 100
    norm_df = norm_df.merge(temp[['date', f'{t}_norm']], on='date', how='left')

norm_df['msft_minus_qqq'] = norm_df['MSFT_norm'] - norm_df['QQQ_norm']

base = base.drop(columns=['msft_minus_qqq'], errors='ignore')
base = base.merge(
    norm_df[['date', 'msft_minus_qqq']],
    on='date',
    how='left'
)

# -----------------------------
# 6) Safe z-score helper
# -----------------------------
def safe_zscore(series):
    std = series.std(ddof=0)
    if std == 0 or pd.isna(std):
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - series.mean()) / std

base['return_z'] = safe_zscore(base['msft_return'].fillna(0))
base['tone_z'] = safe_zscore(base['tone'].fillna(0))
base['views_z'] = safe_zscore(base['views'].fillna(0))
base['rel_z'] = safe_zscore(base['msft_minus_qqq'].fillna(0))

# -----------------------------
# 7) Data-driven peak / non-peak labels
# -----------------------------
# Peak attention = top 20% of pageview days
attention_threshold = base['views'].quantile(0.80)
base['attention_group'] = np.where(
    base['views'] >= attention_threshold,
    'Peak Attention',
    'Non-Peak Attention'
)

# Low tone = bottom 20% of tone days
tone_threshold = base['tone'].quantile(0.20)
base['tone_group'] = np.where(
    base['tone'] <= tone_threshold,
    'Low Tone',
    'Normal Tone'
)

# Relative performance groups
high_rel_threshold = base['msft_minus_qqq'].quantile(0.80)
low_rel_threshold = base['msft_minus_qqq'].quantile(0.20)

base['relative_group'] = np.select(
    [
        base['msft_minus_qqq'] >= high_rel_threshold,
        base['msft_minus_qqq'] <= low_rel_threshold
    ],
    [
        'MSFT Outperformed QQQ',
        'MSFT Underperformed QQQ'
    ],
    default='MSFT In Line with QQQ'
)

# Return groups
base['return_group'] = np.select(
    [
        base['msft_return'] > 0.5,
        base['msft_return'] < -0.5
    ],
    [
        'Strong Up Day',
        'Strong Down Day'
    ],
    default='Flat / Mild Move'
)

# -----------------------------
# 8) Dynamic subsets for use in visualizations
# -----------------------------
peak_attention_days = base[base['attention_group'] == 'Peak Attention'].copy()
non_peak_attention_days = base[base['attention_group'] == 'Non-Peak Attention'].copy()

low_tone_days = base[base['tone_group'] == 'Low Tone'].copy()
normal_tone_days = base[base['tone_group'] == 'Normal Tone'].copy()

outperform_days = base[base['relative_group'] == 'MSFT Outperformed QQQ'].copy()
underperform_days = base[base['relative_group'] == 'MSFT Underperformed QQQ'].copy()

# -----------------------------
# 9) Professional color palette
# -----------------------------
COLORS = {
    'msft': '#2563EB',
    'qqq': '#64748B',
    'peak': '#0F766E',
    'non_peak': '#CBD5E1',
    'tone_low': '#7C3AED',
    'tone_normal': '#DDD6FE',
    'positive': '#16A34A',
    'negative': '#DC2626',
    'neutral': '#94A3B8',
    'grid': '#E5E7EB',
    'text': '#111827',
    'selected': '#111111'
}

def style_fig(fig, title, height):
    fig.update_layout(
        title=title,
        template='plotly_white',
        height=height,
        margin=dict(l=40, r=30, t=60, b=40),
        paper_bgcolor='white',
        plot_bgcolor='white',
        font=dict(family='Arial', color=COLORS['text']),
        title_font=dict(size=19),
        legend=dict(
            orientation='h',
            y=1.08,
            x=0,
            bgcolor='rgba(255,255,255,0.75)'
        )
    )
    fig.update_xaxes(showgrid=True, gridcolor=COLORS['grid'], zeroline=False)
    fig.update_yaxes(showgrid=True, gridcolor=COLORS['grid'], zeroline=False)
    return fig

# -----------------------------
# 10) Quick checks
# -----------------------------
print("Base columns:")
print(base.columns.tolist())

print("\nWindow:")
print(base['date'].min(), "to", base['date'].max())

print("\nAttention threshold:", attention_threshold)
print("Tone threshold:", tone_threshold)
print("Relative high threshold:", high_rel_threshold)
print("Relative low threshold:", low_rel_threshold)

print("\nCounts:")
print(base['attention_group'].value_counts())
print(base['tone_group'].value_counts())
print(base['relative_group'].value_counts())

print("\nPreview:")
print(base[['date', 'views', 'attention_group', 'tone', 'tone_group', 'msft_minus_qqq', 'relative_group']].head(10))

Base columns:
['date', 'msft_close', 'msft_return', 'tone', 'views', 'views_ma3', 'msft_minus_qqq', 'return_z', 'tone_z', 'views_z', 'rel_z', 'attention_group', 'tone_group', 'relative_group', 'return_group']

Window:
2025-05-01 00:00:00 to 2025-06-13 00:00:00

Attention threshold: 9480.0
Tone threshold: 0.2261
Relative high threshold: 1.5325064671393562
Relative low threshold: -0.5048895626004963

Counts:
attention_group
Non-Peak Attention    24
Peak Attention         7
Name: count, dtype: int64
tone_group
Normal Tone    24
Low Tone        7
Name: count, dtype: int64
relative_group
MSFT In Line with QQQ      17
MSFT Outperformed QQQ       7
MSFT Underperformed QQQ     7
Name: count, dtype: int64

Preview:
        date  views     attention_group    tone   tone_group  msft_minus_qqq  \
0 2025-05-01   8156  Non-Peak Attention  0.4496  Normal Tone        0.000000   
1 2025-05-02   7543  Non-Peak Attention  0.0058     Low Tone        0.838132   
2 2025-05-05   7946  Non-Peak Attention  0.6

In [78]:
# =========================
# FINAL DASH APP (LINKED + CLEAN LAYOUT)
# PASTE THIS AFTER YOUR PREP BLOCK
# =========================

from dash import Dash, dcc, html, Input, Output
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from PIL import Image

# -----------------------------
# Windowed data
# -----------------------------
X_START = pd.to_datetime(WINDOW_START)
X_END   = pd.to_datetime(WINDOW_END)

base_dash = base.copy()
base_dash["date"] = pd.to_datetime(base_dash["date"])
base_dash = base_dash[(base_dash["date"] >= X_START) & (base_dash["date"] <= X_END)].copy()

msft_ohlc_dash = stocks["MSFT"].copy()
msft_ohlc_dash["date"] = pd.to_datetime(msft_ohlc_dash["date"])
msft_ohlc_dash = msft_ohlc_dash[(msft_ohlc_dash["date"] >= X_START) & (msft_ohlc_dash["date"] <= X_END)].copy()

comparison_tickers = ["MSFT", "AAPL", "AMZN", "GOOGL", "META", "ORCL", "CRM", "QQQ", "XLK"]

compare_dash = pd.DataFrame({"date": stocks["MSFT"]["date"]})
compare_dash["date"] = pd.to_datetime(compare_dash["date"])
compare_dash = compare_dash[(compare_dash["date"] >= X_START) & (compare_dash["date"] <= X_END)].copy()

for t in comparison_tickers:
    temp = stocks[t][["date", "close"]].copy().sort_values("date")
    temp["date"] = pd.to_datetime(temp["date"])
    temp = temp[(temp["date"] >= X_START) & (temp["date"] <= X_END)].copy()
    temp[f"{t}_norm"] = temp["close"] / temp["close"].iloc[0] * 100
    compare_dash = compare_dash.merge(temp[["date", f"{t}_norm"]], on="date", how="left")

# recompute signals cleanly
base_dash["views_ma5"] = base_dash["views"].rolling(5, min_periods=1).mean()
base_dash["abs_return"] = base_dash["msft_return"].abs().fillna(0)

views_cutoff = base_dash["views"].quantile(0.80)
base_dash["attention_group"] = np.where(
    base_dash["views"] >= views_cutoff,
    "Peak Attention",
    "Non-Peak Attention"
)

# -----------------------------
# Colors
# -----------------------------
COLORS = {
    "bg": "#07111F",
    "panel": "#0B1728",
    "card": "#0D172A",
    "grid": "rgba(148, 163, 184, 0.12)",
    "text": "#E5E7EB",
    "muted": "#94A3B8",
    "white": "#F8FAFC",
    "msft": "#4EA1FF",
    "up": "#16C784",
    "down": "#EA3943",
    "teal": "#14B8A6",
    "cyan": "#22D3EE",
    "qqq": "#9CA3AF",
    "non_peak": "rgba(148, 163, 184, 0.18)",
    "card_border": "rgba(148,163,184,0.20)",
    "avg": "#14B8A6",
    "median": "#CBD5E1"
}

COMPETITOR_COLORS = {
    "AAPL": "#9CA3AF",
    "AMZN": "#F59E0B",
    "GOOGL": "#60A5FA",
    "META": "#A78BFA",
    "ORCL": "#F87171",
    "CRM": "#38BDF8",
    "QQQ": "#9CA3AF",
    "XLK": "#34D399"
}

# -----------------------------
# Logo
# -----------------------------
logo_candidates = [
    "Microsoft_logo.svg.png",
    "microsoft_logo.svg(1).png",
    "microsoft_logo.svg.png",
    "msft_logo.png"
]

msft_logo = None
for f in logo_candidates:
    if os.path.exists(f):
        msft_logo = Image.open(f)
        break

# -----------------------------
# Helpers
# -----------------------------
def style_dark(fig, title, height=340):
    fig.update_layout(
        title={
            "text": f"<b>{title}</b>",
            "x": 0.5,
            "xanchor": "center",
            "font": {"size": 22, "color": COLORS["white"]}
        },
        template="plotly_dark",
        height=height,
        paper_bgcolor=COLORS["bg"],
        plot_bgcolor=COLORS["panel"],
        font=dict(family="Arial", color=COLORS["text"], size=13),
        margin=dict(l=55, r=55, t=78, b=48),
        legend=dict(
            orientation="h",
            y=1.04,
            x=0,
            bgcolor="rgba(0,0,0,0)",
            font=dict(size=12)
        ),
        hoverlabel=dict(
            bgcolor="#0F172A",
            bordercolor="#334155",
            font=dict(color=COLORS["white"])
        ),
        dragmode="select"
    )

    fig.update_xaxes(
        showgrid=True,
        gridcolor=COLORS["grid"],
        zeroline=False,
        linecolor="rgba(255,255,255,0.15)",
        tickfont=dict(color=COLORS["text"], size=11),
        title_font=dict(color=COLORS["text"], size=14)
    )

    fig.update_yaxes(
        showgrid=True,
        gridcolor=COLORS["grid"],
        zeroline=False,
        linecolor="rgba(255,255,255,0.15)",
        tickfont=dict(color=COLORS["text"], size=11),
        title_font=dict(color=COLORS["text"], size=14)
    )
    return fig

def get_filtered_data(sel_attention=None, sel_candle=None, sel_compare=None):
    selections = [sel_attention, sel_candle, sel_compare]

    start, end = X_START, X_END

    for sel in selections:
        if sel and "range" in sel and "x" in sel["range"]:
            try:
                x0, x1 = sel["range"]["x"]
                start = pd.to_datetime(x0)
                end = pd.to_datetime(x1)
                break
            except Exception:
                pass

    filtered = base_dash[(base_dash["date"] >= start) & (base_dash["date"] <= end)].copy()
    filtered_ohlc = msft_ohlc_dash[(msft_ohlc_dash["date"] >= start) & (msft_ohlc_dash["date"] <= end)].copy()
    filtered_compare = compare_dash[(compare_dash["date"] >= start) & (compare_dash["date"] <= end)].copy()

    return start, end, filtered, filtered_ohlc, filtered_compare

def get_clicked_date(click_attention=None, click_candle=None, click_tone=None, click_compare=None):
    click_sources = [click_attention, click_candle, click_tone, click_compare]

    for click in click_sources:
        if click and "points" in click and len(click["points"]) > 0:
            pt = click["points"][0]

            # first try customdata if available
            if "customdata" in pt and pt["customdata"] is not None:
                try:
                    if isinstance(pt["customdata"], (list, tuple)):
                        return pd.to_datetime(pt["customdata"][0])
                    return pd.to_datetime(pt["customdata"])
                except Exception:
                    pass

            # then try x if x is a date
            if "x" in pt:
                try:
                    return pd.to_datetime(pt["x"])
                except Exception:
                    pass

    return None

# -----------------------------
# Figure builders
# -----------------------------
def make_summary(start, end, filtered, filtered_compare):
    final_price = filtered["msft_close"].iloc[-1]
    start_price = filtered["msft_close"].iloc[0]
    msft_window_return = (final_price / start_price - 1) * 100

    qqq_start = filtered_compare["QQQ_norm"].iloc[0]
    qqq_end = filtered_compare["QQQ_norm"].iloc[-1]
    qqq_window_return = (qqq_end / qqq_start - 1) * 100 if qqq_start != 0 else np.nan

    peak_views_row = filtered.loc[filtered["views"].idxmax()]
    lowest_tone_row = filtered.loc[filtered["tone"].idxmin()]
    best_rel_row = filtered.loc[filtered["msft_minus_qqq"].idxmax()]

    fig = go.Figure()
    fig.update_xaxes(visible=False, range=[0, 1])
    fig.update_yaxes(visible=False, range=[0, 1])

    # centered cards with more even spacing
    cards = [
        {"x0": 0.04, "x1": 0.19, "label": "MSFT CLOSE", "sub": "Microsoft closing price", "value": f"${final_price:,.2f}", "extra": f"{msft_window_return:+.1f}%"},
        {"x0": 0.23, "x1": 0.38, "label": "QQQ RETURN", "sub": "Window return", "value": f"{qqq_window_return:+.2f}%", "extra": ""},
        {"x0": 0.42, "x1": 0.57, "label": "PEAK ATTENTION", "sub": peak_views_row['date'].strftime('%b %d'), "value": f"{peak_views_row['views']/1000:.1f}k", "extra": ""},
        {"x0": 0.61, "x1": 0.76, "label": "LOWEST TONE", "sub": lowest_tone_row['date'].strftime('%b %d'), "value": f"{lowest_tone_row['tone']:.3f}", "extra": ""},
        {"x0": 0.80, "x1": 0.95, "label": "BEST MSFT VS QQQ", "sub": best_rel_row['date'].strftime('%b %d'), "value": f"{best_rel_row['msft_minus_qqq']:.2f} pts", "extra": ""},
    ]

    for c in cards:
        fig.add_shape(
            type="rect",
            xref="paper", yref="paper",
            x0=c["x0"], x1=c["x1"], y0=0.14, y1=0.66,
            line=dict(color=COLORS["card_border"], width=1.2),
            fillcolor=COLORS["card"],
            layer="below"
        )

        xmid = (c["x0"] + c["x1"]) / 2

        fig.add_annotation(
            x=xmid, y=0.59,
            text=f"<b>{c['label']}</b>",
            showarrow=False,
            font=dict(size=12, color=COLORS["white"])
        )
        fig.add_annotation(
            x=xmid, y=0.52,
            text=c["sub"],
            showarrow=False,
            font=dict(size=11, color=COLORS["muted"])
        )
        fig.add_annotation(
            x=xmid, y=0.35,
            text=c["value"],
            showarrow=False,
            font=dict(size=26, color=COLORS["white"])
        )
        if c["extra"]:
            fig.add_annotation(
                x=xmid, y=0.22,
                text=c["extra"],
                showarrow=False,
                font=dict(size=14, color=COLORS["up"] if c["extra"].startswith("+") else COLORS["down"])
            )

    # title + subtitle with more separation
    fig.add_annotation(
        x=0.50, y=0.95,
        text="<b>Microsoft Dashboard</b>",
        showarrow=False,
        font=dict(size=30, color=COLORS["white"]),
        xanchor="center"
    )

    fig.add_annotation(
        x=0.56, y=0.84,
        text=f"Analysis Window: {start.strftime('%b %-d, %Y')} – {end.strftime('%b %-d, %Y')}",
        showarrow=False,
        font=dict(size=14, color=COLORS["muted"]),
        xanchor="center"
    )
    fig.update_layout(
        template="plotly_dark",
        paper_bgcolor=COLORS["bg"],
        plot_bgcolor=COLORS["bg"],
        height=220,
        margin=dict(l=20, r=20, t=8, b=0)
    )
    return fig

def make_candle(filtered, filtered_ohlc, selected_date=None):
    fig = make_subplots(specs=[[{"secondary_y": True}]])

    fig.add_trace(
        go.Candlestick(
            x=filtered_ohlc["date"],
            open=filtered_ohlc["open"],
            high=filtered_ohlc["high"],
            low=filtered_ohlc["low"],
            close=filtered_ohlc["close"],
            increasing_line_color=COLORS["up"],
            decreasing_line_color=COLORS["down"],
            increasing_fillcolor=COLORS["up"],
            decreasing_fillcolor=COLORS["down"],
            name="MSFT",
            customdata=filtered_ohlc["date"].astype(str)
        ),
        secondary_y=False
    )

    fig.add_trace(
        go.Bar(
            x=filtered["date"],
            y=filtered["views"],
            name="Attention",
            marker_color=np.where(filtered["attention_group"] == "Peak Attention", COLORS["teal"], COLORS["non_peak"]),
            opacity=0.16,
            customdata=filtered["date"].astype(str)
        ),
        secondary_y=True
    )

    style_dark(fig, "Microsoft Price Action", 380)
    fig.update_layout(xaxis_rangeslider_visible=False)
    fig.update_yaxes(title_text="MSFT Price ($)", secondary_y=False)
    fig.update_yaxes(title_text="Wikipedia Pageviews", secondary_y=True, showgrid=False)

    if selected_date is not None:
        row = filtered_ohlc[filtered_ohlc["date"] == selected_date]
        if not row.empty:
            fig.add_trace(
                go.Scatter(
                    x=row["date"],
                    y=row["close"],
                    mode="markers",
                    marker=dict(size=16, symbol="circle-open", line=dict(width=2, color=COLORS["white"])),
                    showlegend=False,
                    hoverinfo="skip"
                ),
                secondary_y=False
            )
    return fig
def make_attention(filtered, selected_date=None):
    peak_filtered = filtered[filtered["attention_group"] == "Peak Attention"].copy()
    fig = make_subplots(specs=[[{"secondary_y": True}]])

    fig.add_trace(
        go.Bar(
            x=filtered["date"],
            y=filtered["views"],
            name="Daily Pageviews",
            marker_color=np.where(filtered["attention_group"] == "Peak Attention", COLORS["teal"], COLORS["non_peak"]),
            opacity=0.40,
            customdata=filtered["date"].astype(str)
        ),
        secondary_y=False
    )

    fig.add_trace(
        go.Scatter(
            x=filtered["date"],
            y=filtered["views_ma5"],
            mode="lines",
            name="Pageviews (5-day avg)",
            line=dict(color=COLORS["cyan"], width=2, dash="dot"),
            customdata=filtered["date"].astype(str)
        ),
        secondary_y=False
    )

    fig.add_trace(
        go.Scatter(
            x=peak_filtered["date"],
            y=peak_filtered["msft_return"],
            mode="markers",
            name="MSFT Return on Peak Days",
            marker=dict(
                size=8 + peak_filtered["abs_return"] * 2.5,
                color=np.where(peak_filtered["msft_return"] >= 0, COLORS["up"], COLORS["down"]),
                line=dict(width=1, color=COLORS["bg"])
            ),
            customdata=peak_filtered["date"].astype(str),
            hovertemplate="Date: %{x|%b %d, %Y}<br>MSFT Daily Return: %{y:.2f}%<extra></extra>"
        ),
        secondary_y=True
    )

    fig.add_hline(y=0, line_dash="dash", line_color=COLORS["muted"], line_width=1.2, secondary_y=True)

    style_dark(fig, "Public Attention Signal (Wikipedia Pageviews)", 355)
    fig.update_yaxes(title_text="Wikipedia Pageviews", secondary_y=False)
    fig.update_yaxes(title_text="MSFT Daily Return (%)", secondary_y=True, showgrid=False)

    if selected_date is not None:
        row = peak_filtered[peak_filtered["date"] == selected_date]
        if not row.empty:
            fig.add_trace(
                go.Scatter(
                    x=row["date"],
                    y=row["msft_return"],
                    mode="markers",
                    marker=dict(size=18, symbol="circle-open", line=dict(width=2, color=COLORS["white"])),
                    showlegend=False,
                    hoverinfo="skip"
                ),
                secondary_y=True
            )
    return fig

def make_tone(filtered, selected_date=None):
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=filtered["tone"],
            y=filtered["msft_return"],
            mode="markers",
            name="Daily observations",
            marker=dict(
                size=12,
                color=np.where(filtered["msft_return"] >= 0, COLORS["up"], COLORS["down"]),
                opacity=0.85,
                line=dict(width=1, color=COLORS["bg"])
            ),
            text=filtered["date"].dt.strftime("%b %d, %Y"),
            customdata=filtered["date"].astype(str),
            hovertemplate=(
                "Date: %{text}<br>"
                "Media Tone: %{x:.3f}<br>"
                "MSFT Daily Return: %{y:.2f}%<extra></extra>"
            )
        )
    )

    fig.add_hline(y=0, line_dash="dot", line_color="rgba(255,255,255,0.25)", line_width=1)
    fig.add_vline(x=0, line_dash="dot", line_color="rgba(255,255,255,0.25)", line_width=1)

    style_dark(fig, "Media Tone vs MSFT Daily Return", 285)
    fig.update_xaxes(title_text="Media Tone", range=[-0.5, 1.2])
    fig.update_yaxes(title_text="MSFT Daily Return (%)")

    if selected_date is not None:
        row = filtered[filtered["date"] == selected_date]
        if not row.empty:
            fig.add_trace(
                go.Scatter(
                    x=row["tone"],
                    y=row["msft_return"],
                    mode="markers",
                    marker=dict(size=18, symbol="circle-open", line=dict(width=2, color=COLORS["white"])),
                    showlegend=False,
                    hoverinfo="skip"
                )
            )
    return fig

def make_compare(filtered_compare, selected_comp="QQQ", selected_date=None):
    competitor_options = ["AAPL", "AMZN", "GOOGL", "META", "ORCL", "CRM", "QQQ", "XLK"]
    title_text = "MSFT vs Competitors" if selected_comp == "All" else f"MSFT vs {selected_comp}"

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=filtered_compare["date"],
            y=filtered_compare["MSFT_norm"],
            name="MSFT",
            line=dict(color=COLORS["msft"], width=3.5),
            customdata=filtered_compare["date"].astype(str)
        )
    )

    comps_to_show = competitor_options if selected_comp == "All" else [selected_comp]

    for comp in comps_to_show:
        fig.add_trace(
            go.Scatter(
                x=filtered_compare["date"],
                y=filtered_compare[f"{comp}_norm"],
                name=comp,
                line=dict(color=COMPETITOR_COLORS.get(comp, COLORS["qqq"]), width=2.5, dash="dash"),
                customdata=filtered_compare["date"].astype(str)
            )
        )

    style_dark(fig, title_text, 285)
    fig.update_yaxes(title_text="Normalized Index")

    if selected_date is not None:
        row = filtered_compare[filtered_compare["date"] == selected_date]
        if not row.empty:
            fig.add_trace(
                go.Scatter(
                    x=row["date"],
                    y=row["MSFT_norm"],
                    mode="markers",
                    marker=dict(size=16, symbol="circle-open", line=dict(width=2, color=COLORS["white"])),
                    showlegend=False,
                    hoverinfo="skip"
                )
            )
    return fig


# -----------------------------
# Dash app
# -----------------------------
app = Dash(__name__)
app.title = "Microsoft Dashboard"

app.index_string = """
<!DOCTYPE html>
<html>
<head>
    {%metas%}
    <title>{%title%}</title>
    {%favicon%}
    {%css%}
    <style>
        body {
            margin: 0;
            background: #07111F;
            font-family: Arial, sans-serif;
            color: #E5E7EB;
        }
        .container {
            width: min(1450px, 97vw);
            margin: 0 auto;
            padding: 16px 20px 28px 20px;
        }
        .row-2 {
            display: grid;
            grid-template-columns: minmax(0, 1fr) minmax(0, 1fr);
            gap: 24px;
            align-items: start;
            margin-top: 6px;
        }
        @media (max-width: 1100px) {
            .row-2 {
                grid-template-columns: 1fr;
            }
        }
    </style>
</head>
<body>
    {%app_entry%}
    <footer>
        {%config%}
        {%scripts%}
        {%renderer%}
    </footer>
</body>
</html>
"""

app.layout = html.Div(
    className="container",
    children=[
        dcc.Graph(
            id="summary-graph",
            config={"displayModeBar": False},
            style={"marginBottom": "8px"}
        ),

        html.Div(
            "Brush a date range on the Public Attention chart to filter the dashboard. Click a point/day in any chart to highlight that date across views.",
            style={
                "color": "#94A3B8",
                "fontSize": "12px",
                "margin": "0 0 10px 4px"
            }
        ),

        html.Div(
            id="filter-label",
            style={
                "color": "#CBD5E1",
                "fontSize": "13px",
                "margin": "0 0 12px 4px",
                "fontWeight": "600"
            }
        ),

        dcc.Graph(id="candle-graph", style={"marginBottom": "14px"}),
        dcc.Graph(id="attention-graph", style={"marginBottom": "18px"}),

        html.Div(
            className="row-2",
            children=[
                html.Div(
                    [dcc.Graph(id="tone-graph", style={"height": "320px"})],
                    style={"minWidth": 0}
                ),
                html.Div(
                    [
                        html.Div(
                            "Compare MSFT with:",
                            style={
                                "color": "#94A3B8",
                                "fontSize": "12px",
                                "marginBottom": "6px",
                                "marginLeft": "6px"
                            }
                        ),
                        dcc.Dropdown(
                            id="competitor-dropdown",
                            options=[{"label": "All", "value": "All"}] + [{"label": c, "value": c} for c in ["AAPL", "AMZN", "GOOGL", "META", "ORCL", "CRM", "QQQ", "XLK"]],
                            value="QQQ",
                            clearable=False,
                            style={"width": "180px", "color": "#111827", "marginBottom": "6px", "marginLeft": "6px"}
                        ),
                        dcc.Graph(id="compare-graph", style={"height": "320px"})
                    ],
                    style={"minWidth": 0}
                )
            ]
        )
    ]
)

# -----------------------------
# Callback
# -----------------------------
@app.callback(
    Output("summary-graph", "figure"),
    Output("filter-label", "children"),
    Output("candle-graph", "figure"),
    Output("attention-graph", "figure"),
    Output("tone-graph", "figure"),
    Output("compare-graph", "figure"),
    Input("attention-graph", "selectedData"),
    Input("candle-graph", "selectedData"),
    Input("compare-graph", "selectedData"),
    Input("attention-graph", "clickData"),
    Input("candle-graph", "clickData"),
    Input("tone-graph", "clickData"),
    Input("compare-graph", "clickData"),
    Input("competitor-dropdown", "value"),
)
def update_dashboard(
    sel_attention,
    sel_candle,
    sel_compare,
    click_attention,
    click_candle,
    click_tone,
    click_compare,
    competitor_value
):
    start, end, filtered, filtered_ohlc, filtered_compare = get_filtered_data(
        sel_attention, sel_candle, sel_compare
    )

    selected_date = get_clicked_date(
        click_attention, click_candle, click_tone, click_compare
    )

    filter_text = f"Current filter: {start.strftime('%b %d, %Y')} to {end.strftime('%b %d, %Y')}"

    return (
        make_summary(start, end, filtered, filtered_compare),
        filter_text,
        make_candle(filtered, filtered_ohlc, selected_date),
        make_attention(filtered, selected_date),
        make_tone(filtered, selected_date),
        make_compare(filtered_compare, competitor_value, selected_date),
    )
# -----------------------------
# Run in Colab
# -----------------------------
from dash import jupyter_dash
jupyter_dash.infer_jupyter_proxy_config()

app.run(
    jupyter_mode="external",
    debug=False,
    use_reloader=False,
    port=8050
)

Dash is running on http://127.0.0.1:8050/



INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



 * Serving Flask app '__main__'
 * Debug mode: off


In [77]:
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(8050)"))

https://8050-m-s-kkb-usw3b1-1mci6lm43j19g-b.us-west3-1.prod.colab.dev
